<a href="https://colab.research.google.com/github/ceida/ann_CustomerPurchasePrediction/blob/main/ann_CustomerPurchasePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install tensorflow scikit-learn pandas numpy

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/santoshc1/PowerBI-AI-samples/master/Tutorial_AutomatedML/online_shoppers_intention.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.head())
print("\nTarget distribution (Revenue):")
print(df["Revenue"].value_counts(normalize=True))

(12359, 18)
   Administrative  Administrative_Duration  Informational  \
0               1                 0.000000              2   
1               7               150.357143              1   
2               3                16.000000              3   
3               0                 0.000000              0   
4               0                 0.000000              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                  211.25             144              4627.489571   
1                    9.00             221             11431.001240   
2                   86.00              15              2773.500000   
3                    0.00               7               705.833333   
4                    0.00               7               212.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0     0.001361   0.020664    0.000000         0.0   Nov                 2   
1     0.011149   0.021904    1.582473         

In [4]:
df = df.dropna(subset=["Revenue"]).copy()

rev = df["Revenue"].astype(str).str.strip().str.lower()
df["Revenue_bin"] = rev.map({"true": 1, "false": 0})

print("Revenue NaN after drop:", df["Revenue"].isna().sum())
print("Revenue_bin NaN count:", df["Revenue_bin"].isna().sum())
print(df[["Revenue", "Revenue_bin"]].head(10))

Revenue NaN after drop: 0
Revenue_bin NaN count: 0
   Revenue  Revenue_bin
29   False            0
30   False            0
31   False            0
32   False            0
33   False            0
34   False            0
35   False            0
36   False            0
37   False            0
38   False            0


In [5]:
y = df["Revenue_bin"]
X = df.drop(columns=["Revenue", "Revenue_bin"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Purchase rate:", y.mean())

X shape: (12330, 17)
y shape: (12330,)
Purchase rate: 0.15474452554744525


In [6]:
cat_cols = ["Month", "VisitorType", "Weekend"]

num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical:", cat_cols)
print("Numeric count:", len(num_cols))

Categorical: ['Month', 'VisitorType', 'Weekend']
Numeric count: 14


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ]
)

X_processed = preprocess.fit_transform(X)

print("Yeni veri şekli:", X_processed.shape)

Yeni veri şekli: (12330, 29)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (9864, 29)
Test: (2466, 29)


In [10]:
import tensorflow as tf
from tensorflow.keras import layers

tf.random.set_seed(42)

model = tf.keras.Sequential([
    layers.Input(shape=(29,)),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,505 (5.88 KB)

 Trainable params: 1,505 (5.88 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = {0: weights[0], 1: weights[1]}

print("Class weights:", class_weight)

Class weights: {0: np.float64(0.5915087550971456), 1: np.float64(3.2319790301441675)}


In [14]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8573 - auc: 0.9307 - loss: 0.3318 - val_accuracy: 0.8495 - val_auc: 0.9254 - val_loss: 0.3574
Epoch 2/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8513 - auc: 0.9310 - loss: 0.3294 - val_accuracy: 0.8550 - val_auc: 0.9262 - val_loss: 0.3512
Epoch 3/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8522 - auc: 0.9339 - loss: 0.3234 - val_accuracy: 0.8632 - val_auc: 0.9256 - val_loss: 0.3386
Epoch 4/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8586 - auc: 0.9329 - loss: 0.3245 - val_accuracy: 0.8561 - val_auc: 0.9256 - val_loss: 0.3373
Epoch 5/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8505 - auc: 0.9293 - loss: 0.3311 - val_accuracy: 0.8535 - val_auc: 0.9258 - val_loss: 0.3454
Epoch 6/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8536 - auc: 0.9309 - loss: 0.3272 - val_accuracy: 0.8576 - val_auc: 0.9260 - val_loss: 0.3347
Epoch 7/50
247/247 ━━━━━━━━━━━━━━━━━━━━ 

In [15]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)
print("Test AUC (Keras):", test_auc)

y_prob = model.predict(X_test, verbose=0).ravel()
print("Test AUC (sklearn):", roc_auc_score(y_test, y_prob))

y_pred = (y_prob >= 0.5).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=3))

Test Accuracy: 0.8580697774887085
Test AUC (Keras): 0.9171127676963806
Test AUC (sklearn): 0.9171121785531248

Confusion Matrix:
 [[1798  286]
 [  64  318]]

Classification Report:
               precision    recall  f1-score   support

           0      0.966     0.863     0.911      2084
           1      0.526     0.832     0.645       382

    accuracy                          0.858      2466
   macro avg      0.746     0.848     0.778      2466
weighted avg      0.898     0.858     0.870      2466

